## 创建项目文件夹

::: {.callout-tip}
### 提示词

- 在当前目录下新建如下文件夹-若已存在则忽略：
  - 'data_raw', 'data_clean' 两个文件夹，分别存放原始数据和清洗后的数据。
  - 'output' 文件夹，存放分析结果。
  - 'code' 文件夹，存放数据清洗和分析的代码。
- 添加必要的 Python 头部代码和注释，确保代码可读性和可维护性。
- 在代码中使用适当的函数和库（如 pandas、numpy 等）进行数据清洗和分析。
:::


In [1]:
import os

# Define the parent directory (one level up from current directory)
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# List of folders to create
folders = ['data_raw', 'data_clean', 'output', 'code']

# Create folders if they don't exist
for folder in folders:
    folder_path = os.path.join(parent_dir, folder)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Folder '{folder}' {'already exists' if os.path.exists(folder_path) else 'created'} at: {folder_path}")

print("\nAll folders are ready.")

Folder 'data_raw' already exists at: g:\dsfin\Lecture\data_clean\data_raw
Folder 'data_clean' already exists at: g:\dsfin\Lecture\data_clean\data_clean
Folder 'output' already exists at: g:\dsfin\Lecture\data_clean\output
Folder 'code' already exists at: g:\dsfin\Lecture\data_clean\code

All folders are ready.


## 产生虚构数据

生成第三章「数据清洗」演示用的虚构数据集
执行后在 `../data_raw/` 文件夹生成以下文件：

- `BasicInf.csv`：包含公司基本信息，如股票代码、公司名称、所属行业和是否为国有企业。
- `FinRatio.csv`：包含公司财务比率信息，如杠杆率、公司规模、资产回报率和研发投入比例。
- `Announcement.csv`：包含公司公告文本信息，用于演示文本数据处理。

In [1]:
"""
create_sample_data.py
生成第三章「数据清洗」演示用的虚构数据集
执行后在 data_raw/ 文件夹生成以下文件：
  - BasicInf.csv
  - FinRatio.csv
  - Announcement.csv
"""

import pandas as pd
import os

#os.makedirs('../data_raw', exist_ok=True)


# ── BasicInf ──────────────────────────────────────────────
# 植入问题：stkcd 第二行为整数 2（格式不一致）
#           industry 第三行含前导空格（格式不一致）

basicinf = pd.DataFrame({
    'stkcd':      ['000001', 2,        '000063',  '600028', '600036'],
    'comp_name':  ['平安银行', '万科A', '中兴通讯', '中国石化', '招商银行'],
    'industry':   ['银行业',  '房地产业', ' 通信设备', '石油化工', '银行业'],
    'soe':        [0,          0,         0,          1,         0],
})

basicinf.to_csv('data_raw/BasicInf.csv', index=False, encoding='utf-8-sig')
print("BasicInf.csv 已生成，shape:", basicinf.shape)
print(basicinf.to_string(index=False))
print()


# ── FinRatio ──────────────────────────────────────────────
# 保留变量：Leverage, Size, ROA, RD（删去 CFlow, Cash_holding）
# 实际分析中会放入更多控制变量，这里为演示目的仅保留四个财务指标
#
# 植入问题：
#   - stkcd=2（格式不一致，与 BasicInf 呼应）
#   - 000001 同时含 report_type A 和 B（报表类型混入）
#   - Leverage=3.21（离群值）
#   - ROA=-0.18（离群值）
#   - Size='23.9'（数值存为字符串）
#   - RD 含缺失值（NaN）

finratio = pd.DataFrame({
    'stkcd': [
        '000001','000001','000001','000001',
        2, 2, 2, 2,
        '000063','000063',
    ],
    'year': [
        2022, 2022, 2023, 2023,
        2022, 2022, 2023, 2023,
        2022, 2023,
    ],
    'report_type': [
        'A','B','A','B',
        'A','B','A','B',
        'A','A',
    ],
    'Leverage': [
        0.89, 0.91, 0.88, 0.90,
        0.65, 0.67, 0.68, 0.69,
        0.52, 3.21,        # 3.21 为离群值
    ],
    'Size': [
        25.3, 25.1, 25.4, 25.2,
        24.1, 24.0, 24.2, 24.1,
        23.8, '23.9',      # '23.9' 为字符串类型
    ],
    'ROA': [
        0.012, 0.010, 0.013, 0.011,
        0.045, 0.043, 0.041, 0.040,
        0.038, -0.18,      # -0.18 为离群值
    ],
    'RD': [
        None, None, None, None,   # 银行业通常无研发支出披露
        0.008, 0.008, 0.009, 0.009,
        0.052, 0.048,
    ],
})

finratio.to_csv('data_raw/FinRatio.csv', index=False, encoding='utf-8-sig')
print("FinRatio.csv 已生成，shape:", finratio.shape)
print(finratio.to_string(index=False))
print()


# ── Announcement ─────────────────────────────────────────
# 每行为一条公告原文
# 植入问题：
#   - 前两行完全重复（同一条公告被爬取两次）
#   - 第三条与第一条同一公司同一年，但为另一笔贷款（多条记录合并铺垫）
#   - stkcd=2（格式不一致）
#   - 第三条公告表述方式与前两条有差异（体现非结构化特点）

announcement = pd.DataFrame({
    'stkcd': ['000001', '000001', '000001', 2,        '000063'],
    'year':  [2023,      2023,     2023,     2022,      2023],
    'text': [
        # 第一条（标准格式）
        '本公司于2023年6月1日，与建行深圳分行签署贷款协议，'
        '贷款金额为2亿元，期限3年，利率为年化4.2%。',

        # 第二条（与第一条完全重复，模拟重复爬取）
        '本公司于2023年6月1日，与建行深圳分行签署贷款协议，'
        '贷款金额为2亿元，期限3年，利率为年化4.2%。',

        # 第三条（同公司同年另一笔贷款，表述风格不同）
        # 差异点：金额用万元、银行写全称+总行直贷、
        #         利率写法为「执行利率」、期限写「60个月」
        '经董事会审议通过，本公司与中国工商银行股份有限公司总行'
        '于2023年8月15日签订《流动资金借款合同》，'
        '借款金额15,000万元，借款期限60个月，'
        '执行利率为年利率4.50%，资金用途为补充流动资金。',

        # 第四条（万科A，金额单位亿元，利率较低体现国有大行优势）
        '本公司于2022年3月10日与中国银行股份有限公司总行签署授信协议，'
        '授信额度5亿元，贷款期限3年，贷款年利率3.8%。',

        # 第五条（中兴通讯，农行地级分行，金额单位万元）
        '2023年11月20日，本公司与中国农业银行股份有限公司深圳宝安支行'
        '签订借款合同，金额人民币8,000万元，期限2年，'
        '年化利率4.60%，用于项目建设。',
    ],
})

announcement.to_csv('data_raw/Announcement.csv', index=False, encoding='utf-8-sig')
print("Announcement.csv 已生成，shape:", announcement.shape)
print(announcement.to_string(index=False))

BasicInf.csv 已生成，shape: (5, 4)
 stkcd comp_name industry  soe
000001      平安银行      银行业    0
     2       万科A     房地产业    0
000063      中兴通讯     通信设备    0
600028      中国石化     石油化工    1
600036      招商银行      银行业    0

FinRatio.csv 已生成，shape: (10, 7)
 stkcd  year report_type  Leverage  Size    ROA    RD
000001  2022           A      0.89  25.3  0.012   NaN
000001  2022           B      0.91  25.1  0.010   NaN
000001  2023           A      0.88  25.4  0.013   NaN
000001  2023           B      0.90  25.2  0.011   NaN
     2  2022           A      0.65  24.1  0.045 0.008
     2  2022           B      0.67  24.0  0.043 0.008
     2  2023           A      0.68  24.2  0.041 0.009
     2  2023           B      0.69  24.1  0.040 0.009
000063  2022           A      0.52  23.8  0.038 0.052
000063  2023           A      3.21  23.9 -0.180 0.048

Announcement.csv 已生成，shape: (5, 3)
 stkcd  year                                                                                                text
000001 